# 01 · Encode keyframes with SigLIP2

Produces `derived/embeddings/siglip/*.npy` — the visual index for search.

Row *i* of the output is the frame with `gid == i` in `catalog.parquet`. Shards are
written as they finish, so a disconnected runtime resumes rather than restarting.

**Before running:** build the catalog locally (`aic build-catalog`) and upload
`catalog.parquet` to Drive, so gid ordering is identical on both machines.

In [ ]:
# --- Colab setup -------------------------------------------------------------
# Runtime > Change runtime type > T4 GPU before running.
!nvidia-smi -L
!git clone -q https://github.com/YOUR_ORG/new_aic2026.git /content/aic || (cd /content/aic && git pull -q)
%cd /content/aic
!pip install -q pandas pyarrow pillow tqdm
import sys; sys.path.insert(0, "/content/aic/src")

In [ ]:
# --- Mount Drive -------------------------------------------------------------
# Keyframes go in, artifacts come out. Keeping both on Drive means an interrupted
# runtime resumes instead of restarting from zero.
from google.colab import drive

drive.mount('/content/drive')

from pathlib import Path

DATA = Path('/content/drive/MyDrive/aic2026')
KEYFRAMES = DATA / 'raw/keyframes'
DERIVED   = DATA / 'derived'
DERIVED.mkdir(parents=True, exist_ok=True)
print('keyframes:', KEYFRAMES, KEYFRAMES.exists())

In [ ]:
!pip install -q transformers torch --upgrade

import torch
from transformers import AutoImageProcessor, AutoModel

MODEL_ID = "google/siglip2-so400m-patch14-384"
DIM = 1152

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device).eval()
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
print(model.config.name_or_path, "on", device)

In [ ]:
import numpy as np


def encode(images):
    """PIL images -> L2-normalized float32 vectors."""
    batch = processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():
        features = model.get_image_features(pixel_values=batch["pixel_values"].half())
    vectors = features.float().cpu().numpy()
    return vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

from aic.index.embed import embed_catalog

catalog = pd.read_parquet(DERIVED / 'catalog.parquet')
print(f'{len(catalog):,} frames across {catalog.video_id.nunique()} videos')

bar = tqdm(total=len(catalog))
embed_catalog(
    catalog=catalog,
    encode_fn=encode,
    output_dir=DERIVED / 'embeddings/siglip',
    dim=DIM,
    batch_size=64,
    shard_size=20_000,
    repo_root=DATA,                    # catalog paths are relative to the data root
    progress=lambda done, total: bar.update(done - bar.n),
)
bar.close()

## Download

Copy `derived/embeddings/siglip/` to `data/derived/embeddings/siglip/` locally, then:

```bash
# set embedding.active: siglip in configs/default.yaml first
aic build-index
```